# RAG para cultivos de alto valor agregado.
**TCC — BI Master PUC-Rio**

## Rafael Keniti Gerbasi Nishioka

## Matrícula: 232100394
## Turma: 2023 / 3

Base de conhecimento: 16 documentos oficiais da Embrapa (pragas, doenças fúngicas/bacterianas e viroses) para tomate, morango e pimenta.


## 1. Instalação e Imports

In [ ]:
!pip install -q groq chromadb pypdf pymupdf sentence-transformers gradio

import os
import json
from datetime import datetime

from google.colab import userdata
from google.colab import files

import pypdf
import fitz  # pymupdf - fallback para PDFs com encoding problematico

from sentence_transformers import SentenceTransformer
import chromadb

from groq import Groq

import gradio as gr

## 2. Chave Groq

API key em **Colab > Secrets** com o nome `GROQ_API_KEY`, com o toggle **Notebook access** ativado.
Ou substitua 'chave_API'.

In [ ]:
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('Chave Groq carregada.')
except Exception:
    os.environ['GROQ_API_KEY'] = 'chave_API'

## 3. Upload dos PDFs da Embrapa (16 documentos)

Carregar de todos os PDFs

In [ ]:
# Limpa pasta de uploads anteriores para evitar duplicatas com sufixo (1), (2)
!rm -rf /content/embrapa_pdfs
os.makedirs('/content/embrapa_pdfs', exist_ok=True)

print('Faca upload dos PDFs da Embrapa:')
uploaded = files.upload()

for filename, content in uploaded.items():
    with open(f'/content/embrapa_pdfs/{filename}', 'wb') as f:
        f.write(content)
    print(f'Salvo: {filename}')

pdfs = [f for f in os.listdir('/content/embrapa_pdfs') if f.endswith('.pdf')]
print(f'\nTotal de PDFs carregados: {len(pdfs)}')

## 4. Processamento e Extração de Texto (com fallback automático)

In [ ]:
fonte_map = {
    # TOMATE
    'DOC-192-Final.pdf':                 'Embrapa - Manejo Integrado de Pragas do Tomate (2022)',
    'ct100.pdf':                         'Embrapa - Doencas do Tomateiro em Cultivo Protegido',
    'COT110.pdf':                        'Embrapa - Vira-cabeca do Tomateiro',
    'CT142.pdf':                         'Embrapa - Viroses Transmitidas por Mosca-branca no Tomateiro (Geminivirus/Crinivirus)',
    'manejo_eficiente_de_tripes_e_virus_associados_em_cultivos_de_tomate_e_pepino.pdf':
                                          'Embrapa - Manejo Eficiente de Tripes e Virus Associados em Tomate e Pepino',

    # PIMENTAO
    'DOC-176.pdf':                       'Embrapa - Guia de Identificacao de Pragas do Pimentao (2020)',
    'cot132.pdf':                        'Embrapa - Controle das Principais Doencas do Pimentao',

    # PIMENTA (Capsicum chinense/baccatum/frutescens)
    'ct115.pdf':                         'Embrapa - Manejo Integrado de Pragas de Pimentas do Genero Capsicum',
    'cot35.pdf':                         'Embrapa - Recomendacoes de Manejo da Antracnose do Pimentao e das Pimentas',
    'CT169.pdf':                         'Embrapa - Principais Viroses que Afetam a Pimenteira (Capsicum spp.)',

    # MORANGO
    'ct40.pdf':                          'Embrapa - Descricao e Manejo das Principais Pragas do Morangueiro (CT40)',
    'ct90.pdf':                          'Embrapa - Descricao e Manejo das Principais Pragas do Morangueiro (CT90)',
    'Sistema-de-Producao-do-Morango.pdf':'Embrapa - Sistema de Producao do Morango (doencas fungicas)',
    '14136.pdf':                         'Embrapa - Principais Viroses do Morangueiro e seu Controle',

    # PRAGA TRANSVERSAL - TRIPES
    'MOURA930001.pdf':                   'Embrapa - Tripes: A Praga Silenciosa',

    # CONTROLE BIOLOGICO (todas as culturas)
    'DOC-169-Internet-2.pdf':            'Embrapa - Guia de Inimigos Naturais em Cultivos de Hortalicas',
}


def extrair_com_pypdf(filepath):
    paginas = []
    with open(filepath, 'rb') as f:
        reader = pypdf.PdfReader(f)
        for i, page in enumerate(reader.pages):
            texto = page.extract_text()
            paginas.append({'texto': texto or '', 'pagina': i + 1})
    return paginas


def extrair_com_pymupdf(filepath):
    paginas = []
    doc = fitz.open(filepath)
    for i, page in enumerate(doc):
        texto = page.get_text()
        paginas.append({'texto': texto or '', 'pagina': i + 1})
    doc.close()
    return paginas


def texto_parece_corrompido(texto, amostra=500):
    trecho = texto[:amostra].lower()
    if len(trecho.strip()) < 30:
        return False
    vogais_e_espaco = sum(trecho.count(c) for c in 'aeiou ')
    proporcao = vogais_e_espaco / max(len(trecho), 1)
    return proporcao < 0.15


def extrair_texto_pdf(filepath, nome_arquivo):
    paginas = extrair_com_pypdf(filepath)
    texto_amostra = ' '.join(p['texto'] for p in paginas[:3])
    if texto_parece_corrompido(texto_amostra):
        print(f'  [!] Texto suspeito em {nome_arquivo} via pypdf. Tentando pymupdf...')
        paginas_fb = extrair_com_pymupdf(filepath)
        texto_amostra_fb = ' '.join(p['texto'] for p in paginas_fb[:3])
        if not texto_parece_corrompido(texto_amostra_fb):
            print(f'  [OK] pymupdf resolveu a extracao de {nome_arquivo}.')
        else:
            print(f'  [AVISO] Ambos os metodos suspeitos para {nome_arquivo}. Usando pymupdf.')
        return paginas_fb
    return paginas


all_docs_raw = []
for filename in os.listdir('/content/embrapa_pdfs'):
    if not filename.endswith('.pdf'):
        continue
    fonte = fonte_map.get(filename, filename)
    paginas = extrair_texto_pdf(f'/content/embrapa_pdfs/{filename}', filename)
    for pag in paginas:
        if pag['texto'].strip():
            all_docs_raw.append({
                'texto': pag['texto'],
                'fonte': fonte,
                'arquivo': filename,
                'pagina': pag['pagina']
            })
    print(f'{filename}: {len(paginas)} paginas processadas')

print(f'\nTotal de paginas com texto extraido: {len(all_docs_raw)}')

## 5. Chunking

In [ ]:
def chunkar(texto, chunk_size=1000, overlap=150):
    chunks = []
    start = 0
    while start < len(texto):
        chunks.append(texto[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in all_docs_raw:
    for chunk in chunkar(doc['texto']):
        if len(chunk.strip()) > 100:
            all_chunks.append({
                'texto': chunk,
                'fonte': doc['fonte'],
                'arquivo': doc['arquivo'],
                'pagina': doc['pagina']
            })

print(f'Total de chunks: {len(all_chunks)}')
print(f'\nExemplo de chunk:')
print(all_chunks[10]['texto'][:300])
print(f"Fonte: {all_chunks[10]['fonte']}")

## 6. Embeddings e Indexação no ChromaDB

In [ ]:
print('Carregando modelo de embedding')
embed_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print('Modelo carregado.')

print('Indexando no ChromaDB...')
client = chromadb.Client()

# Remove colecao anterior, se existir, para evitar mistura com base antiga
try:
    client.delete_collection('fitossanitario')
except Exception:
    pass
collection = client.create_collection('fitossanitario')

textos = [c['texto'] for c in all_chunks]
ids = [str(i) for i in range(len(all_chunks))]
metadatas = [{'fonte': c['fonte'], 'arquivo': c['arquivo'], 'pagina': str(c['pagina'])} for c in all_chunks]

batch_size = 100
for i in range(0, len(textos), batch_size):
    batch_textos = textos[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]
    batch_meta = metadatas[i:i+batch_size]
    embeddings = embed_model.encode(batch_textos).tolist()
    collection.add(documents=batch_textos, embeddings=embeddings, ids=batch_ids, metadatas=batch_meta)
    print(f'  Indexados {min(i+batch_size, len(textos))}/{len(textos)} chunks...')

print(f'Indexacao concluida: {len(all_chunks)} chunks')

resultados_teste = collection.query(
    query_embeddings=embed_model.encode(['principais pragas do tomateiro']).tolist(),
    n_results=3
)
print('\nTeste de recuperacao:')
for i, (doc, meta) in enumerate(zip(resultados_teste['documents'][0], resultados_teste['metadatas'][0])):
    print(f"  {i+1}. {meta['fonte']} | p.{meta['pagina']}")
    print(f'     {doc[:150]}...')

## 7. LLM via Groq e Função RAG

In [ ]:
groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])

MODELO_LLM = 'llama-3.3-70b-versatile'

PROMPT_SISTEMA = """Voce e um agronomo especialista em fitossanidade de culturas protegidas,
com foco em tomate, morango e pimenta.
Responda a pergunta com base EXCLUSIVAMENTE no contexto tecnico fornecido,
extraido de publicacoes oficiais da Embrapa.

Regras:
- Seja claro e objetivo, como um agronomo conversando com um produtor
- Cite a fonte ao final da resposta
- Se o contexto nao tiver informacao suficiente, diga explicitamente
- Nao invente informacoes que nao estejam no contexto
- Mencione condicoes que favorecem a praga quando relevante
- Inclua opcoes de controle biologico, cultural e quimico quando disponivel

IMPORTANTE: cite APENAS o documento Embrapa de origem (nome e pagina) que sera fornecido a parte.
NUNCA cite referencias bibliograficas, autores ou trabalhos terceiros que aparecam dentro do texto do contexto - essas sao citacoes dos autores originais, nao fontes que voce deve reproduzir."""


def recuperar_contexto(pergunta, n=5):
    embedding = embed_model.encode([pergunta]).tolist()
    resultados = collection.query(query_embeddings=embedding, n_results=n)
    contexto = ''
    fontes = []
    for doc, meta in zip(resultados['documents'][0], resultados['metadatas'][0]):
        contexto += f"\n---\n{doc}"
        fontes.append(f"{meta['fonte']} p.{meta['pagina']}")
    return contexto, fontes


def consultar_rag(pergunta):
    contexto, fontes = recuperar_contexto(pergunta)
    prompt_usuario = f"""CONTEXTO TECNICO:
{contexto}

PERGUNTA DO PRODUTOR:
{pergunta}

RESPOSTA DO AGRONOMO:"""

    resposta = groq_client.chat.completions.create(
        model=MODELO_LLM,
        messages=[
            {'role': 'system', 'content': PROMPT_SISTEMA},
            {'role': 'user', 'content': prompt_usuario}
        ],
        temperature=0.1,
        max_tokens=1024
    )
    return resposta.choices[0].message.content, fontes


def resposta_sem_rag(pergunta):
    resposta = groq_client.chat.completions.create(
        model=MODELO_LLM,
        messages=[
            {'role': 'system', 'content': 'Voce e um agronomo especialista em fitossanidade. Responda com base no seu conhecimento geral.'},
            {'role': 'user', 'content': pergunta}
        ],
        temperature=0.1,
        max_tokens=1024
    )
    return resposta.choices[0].message.content


print(f'RAG pronto. Modelo: {MODELO_LLM}')

## 8. Testes do RAG

In [ ]:
def mostrar_resposta(pergunta):
    print(f'\nPergunta: {pergunta}')
    print('-' * 60)
    resposta, fontes = consultar_rag(pergunta)
    print('Resposta:')
    print(resposta)
    print('\nFontes consultadas:')
    for f in set(fontes):
        print(f'  - {f}')
    return resposta, fontes


mostrar_resposta('Como identificar e controlar o acaro rajado no tomateiro?')
mostrar_resposta('Quais sao os sintomas de tripes no morangueiro e como fazer o manejo?')
mostrar_resposta('Pulgoes estao atacando meu pimentao. Quais as opcoes de controle?')
mostrar_resposta('Quais inimigos naturais posso usar no controle biologico em hortalicas?')
mostrar_resposta('Como prevenir problemas com tomate no inverno?')

## 9. Avaliação Comparativa — RAG vs Sem Contexto

RAG com a base da Embrapa produz respostas mais precisas, especificas e citaveis do que o LLM sem contexto.

In [ ]:
def comparar(pergunta):
    print('\n' + '=' * 70)
    print(f'PERGUNTA: {pergunta}')
    print('=' * 70)

    print('\n[SEM RAG - conhecimento geral do modelo]')
    sem = resposta_sem_rag(pergunta)
    print(sem)

    print('\n[COM RAG - base Embrapa]')
    com, fontes = consultar_rag(pergunta)
    print(com)
    print('\nFontes RAG:')
    for f in set(fontes):
        print(f'  - {f}')

    return {'pergunta': pergunta, 'sem_rag': sem, 'com_rag': com, 'fontes': list(set(fontes))}


casos = [
    'Como identificar e controlar o acaro rajado no tomateiro?',
    'Quais fungicidas sao recomendados para oidio no morango no Brasil?',
    'O que e manejo integrado de pragas no cultivo de pimentao?',
    'Quais condicoes climaticas favorecem mosca-branca no tomate?',
    'Como prevenir problemas com tomate no inverno?',
]

resultados = [comparar(c) for c in casos]

## 10. Interface de Demonstração (Gradio)

In [ ]:
def interface(pergunta, mostrar_fontes):
    if not pergunta.strip():
        return 'Digite uma pergunta.', ''
    resposta, fontes = consultar_rag(pergunta)
    fontes_texto = ''
    if mostrar_fontes:
        fontes_texto = '\n'.join(set(fontes))
    return resposta, fontes_texto


gr.Interface(
    fn=interface,
    inputs=[
        gr.Textbox(label='Pergunta', placeholder='Ex: Como controlar acaro no tomate?', lines=3),
        gr.Checkbox(label='Mostrar fontes', value=True)
    ],
    outputs=[
        gr.Textbox(label='Resposta do Agronomo Virtual', lines=10),
        gr.Textbox(label='Fontes Embrapa', lines=4)
    ],
    title='RAG para Cultivos Protegidos de Alto Valor',
    description='Sistema de diagnostico baseado em publicacoes oficiais da Embrapa (tomate, morango, pimentao).',
    examples=[
        ['Como identificar e controlar o acaro rajado no tomateiro?', True],
        ['Quais sao os sintomas de tripes no morangueiro?', True],
        ['Pulgoes no pimentao: como controlar?', True],
        ['Como prevenir problemas com tomate no inverno?', True],
    ],
    theme=gr.themes.Soft()
).launch(share=True)

## 11. Exportar Resultados

In [ ]:
output = {
    'data': datetime.now().isoformat(),
    'modelo_llm': MODELO_LLM,
    'embedding': 'paraphrase-multilingual-MiniLM-L12-v2',
    'vectorstore': 'ChromaDB',
    'fontes': list(fonte_map.values()),
    'total_chunks': len(all_chunks),
    'resultados': resultados
}

with open('/content/resultados_rag.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

files.download('/content/resultados_rag.json')
print('Resultados exportados.')